# Monte Carlo Methods — Blackjack

Monte Carlo methods for episodic MDPs: policy evaluation and control without a model of $T$ and $R$.

## Goal
Learn $v_\pi$ / $q_\pi$ and improve policies **without** knowing $T$ and $R$, using complete episodes only (no bootstrapping).

## Environment
Gymnasium `Blackjack-v1` with `sab=True` (Sutton & Barto rules).  
State $(p, d, a)$ = player sum, dealer showing card, usable ace.  
Actions: `0` = stick, `1` = hit. Discount $\gamma = 1$.

## Implemented below
1. Episode generation + fixed baseline policy
2. First-visit / every-visit MC evaluation of $v_\pi$
3. Value-function plots
4. Monte Carlo Exploring Starts (control)
5. On-policy MC control ($\varepsilon$-greedy)
6. Off-policy MC control (weighted importance sampling)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

np.random.seed(42)


## Blackjack environment

Observation is a tuple `(player_sum, dealer_card, usable_ace)`.  
Rewards are typically $0$ mid-episode and $+1$ / $0$ / $-1$ at the end of the hand.


In [ ]:
env = gym.make("Blackjack-v1", sab=True)
obs, info = env.reset()

done = False
while not done:
    action = env.action_space.sample()  # random hit/stick
    next_obs, reward, terminated, truncated, info = env.step(action)
    print(obs, "->", action, "r=", reward, "next=", next_obs, "done=", terminated)
    obs = next_obs
    done = terminated or truncated
print("Final reward:", reward)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)


## Episode utilities

**`generate_episode`.** Plays one full hand under a callable `policy(obs) -> action`, and returns the trajectory as a list of `(s, a, r)`.

**`simple_policy`.** Fixed baseline used for policy *evaluation*: hit if player sum $< 20$, otherwise stick.


In [ ]:
def generate_episode(env, policy):
    """policy(obs) -> action. Returns list of (s, a, r)."""
    episode = []
    obs, info = env.reset()
    done = False
    while not done:
        action = policy(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        episode.append((obs, action, reward))
        obs = next_obs
        done = terminated or truncated
    return episode


def simple_policy(obs):
    player_sum, dealer, usable_ace = obs
    return 1 if player_sum < 20 else 0  # 1=hit, 0=stick


generate_episode(env, simple_policy)


## First-visit Monte Carlo evaluation

Estimate $v_\pi(s)=\mathbb{E}_\pi[G_t \mid S_t=s]$ from episodes generated by a **fixed** policy $\pi$.

Walk each episode **backward** with $G_t = R_{t+1} + \gamma G_{t+1}$, and update $V(s)$ only on the **first** visit to $s$ in that episode.


In [ ]:
def Monte_Carlo_first_visit(env, policy, num_episodes, gamma=1.0):
    # V[player_sum, dealer_card, usable_ace]
    V = np.zeros((32, 11, 2))
    counts = np.zeros((32, 11, 2))

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)

        G = 0.0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = gamma * G + r

            # first-visit: s does not appear earlier in the episode
            earlier_states = [episode[i][0] for i in range(t)]
            if s not in earlier_states:
                player, dealer, ace = s
                counts[player, dealer, ace] += 1
                V[player, dealer, ace] += (G - V[player, dealer, ace]) / counts[player, dealer, ace]

    return V


## Every-visit Monte Carlo evaluation

Same as first-visit, but every occurrence of $s$ in the episode contributes a sample. Samples are correlated, but the estimate still converges to $v_\pi$.


In [ ]:
def Monte_Carlo_every_visit(env, policy, num_episodes, gamma=1.0):
    V = np.zeros((32, 11, 2))
    counts = np.zeros((32, 11, 2))

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)

        G = 0.0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = gamma * G + r

            player, dealer, ace = s
            counts[player, dealer, ace] += 1
            V[player, dealer, ace] += (G - V[player, dealer, ace]) / counts[player, dealer, ace]

    return V


## Plotting $V$ and policies

Heatmaps of player sum ($12$–$21$) × dealer card ($1$–$10$), with / without usable ace.  
For values, color ≈ expected return (red better, blue worse). For policies, blue ≈ stick, red ≈ hit.


In [ ]:
def plot_value_function(V, title=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    titles = ["No usable ace", "Usable ace"]

    for ace, ax in enumerate(axes):
        # rows: player 12..21, cols: dealer 1..10
        grid = V[12:22, 1:11, ace]

        im = ax.imshow(
            grid,
            cmap="coolwarm",
            origin="lower",
            vmin=-1,
            vmax=1,
            aspect="auto",
        )
        ax.set_title(titles[ace])
        ax.set_xlabel("Dealer showing")
        ax.set_ylabel("Player sum")
        ax.set_xticks(range(10))
        ax.set_xticklabels(range(1, 11))
        ax.set_yticks(range(10))
        ax.set_yticklabels(range(12, 22))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if title:
        plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_policy(policy, title="Learned policy"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    titles = ["No usable ace", "Usable ace"]

    for ace, ax in enumerate(axes):
        # 0 = stick, 1 = hit
        grid = policy[12:22, 1:11, ace]

        im = ax.imshow(
            grid,
            cmap="coolwarm",
            origin="lower",
            vmin=0,
            vmax=1,
            aspect="auto",
        )
        ax.set_title(titles[ace])
        ax.set_xlabel("Dealer showing")
        ax.set_ylabel("Player sum")
        ax.set_xticks(range(10))
        ax.set_xticklabels(range(1, 11))
        ax.set_yticks(range(10))
        ax.set_yticklabels(range(12, 22))

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=[0, 1])
        cbar.ax.set_yticklabels(["Stick", "Hit"])

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_q_value(Q, title=None):
    # V(s) = max_a Q(s, a)
    V = np.max(Q, axis=-1)
    plot_value_function(V, title=title)


In [ ]:
env = gym.make("Blackjack-v1", sab=True)
V = Monte_Carlo_first_visit(env, simple_policy, num_episodes=50_000)
print("V[18, 10, 0] =", V[18, 10, 0])  # player 18, dealer 10, no usable ace
plot_value_function(V, title=r"First-visit MC — $v_\pi$ under simple_policy")


## Monte Carlo Exploring Starts (control)

Learn $Q(s,a)$ and a greedy policy without a transition model.

Approximation for Gymnasium: after `reset()`, take a **random first action**, then follow the current greedy policy.  
First-visit updates of $Q$, then $\pi(s) \leftarrow \arg\max_a Q(s,a)$.


In [ ]:
def Monte_Carlo_Exploring_Starts(env, num_episodes, gamma=1.0):
    # Q[player, dealer, ace, action]
    Q = np.zeros((32, 11, 2, 2))
    counts = np.zeros((32, 11, 2, 2))

    # initial policy: stick if sum >= 20, else hit
    policy = np.zeros((32, 11, 2), dtype=int)
    for player in range(32):
        for dealer in range(11):
            for ace in range(2):
                policy[player, dealer, ace] = 0 if player >= 20 else 1

    for _ in range(num_episodes):
        # --- generate one episode with exploring start ---
        episode = []
        s, info = env.reset()
        a = env.action_space.sample()  # random first action (exploring start)
        done = False

        while not done:
            next_s, r, terminated, truncated, info = env.step(a)
            episode.append((s, a, r))
            s = next_s
            done = terminated or truncated
            if not done:
                player, dealer, ace = s
                a = int(policy[player, dealer, ace])  # then follow current π

        # --- first-visit MC update of Q ---
        G = 0.0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = gamma * G + r

            earlier = [(episode[i][0], episode[i][1]) for i in range(t)]
            if (s, a) not in earlier:
                player, dealer, ace = s
                counts[player, dealer, ace, a] += 1
                Q[player, dealer, ace, a] += (
                    G - Q[player, dealer, ace, a]
                ) / counts[player, dealer, ace, a]

                # greedy policy improvement
                policy[player, dealer, ace] = int(np.argmax(Q[player, dealer, ace]))

    return Q, policy


In [ ]:
env = gym.make("Blackjack-v1", sab=True)
Q_es, policy_es = Monte_Carlo_Exploring_Starts(env, num_episodes=200_000)
plot_policy(policy_es, title="MC Exploring Starts — learned policy")
plot_q_value(Q_es, title=r"MC Exploring Starts — $V(s)=\max_a Q(s,a)$")


## On-policy Monte Carlo control ($\varepsilon$-greedy)

Avoid exploring starts: behave with a **soft** policy ($\varepsilon$-greedy) and improve that same policy.

- Generate episodes with $\varepsilon$-greedy w.r.t. the current greedy table
- First-visit update of $Q(s,a)$
- Set $\pi(s) \leftarrow \arg\max_a Q(s,a)$ (exploration stays in the action sampler)


In [ ]:
def Monte_Carlo_on_policy_control(env, num_episodes, gamma=1.0, epsilon=0.1):
    n_actions = env.action_space.n  # 2 for Blackjack
    Q = np.zeros((32, 11, 2, n_actions))
    counts = np.zeros((32, 11, 2, n_actions))

    # greedy action for each state (int: 0=stick, 1=hit)
    policy = np.zeros((32, 11, 2), dtype=int)
    for player in range(32):
        for dealer in range(11):
            for ace in range(2):
                policy[player, dealer, ace] = 0 if player >= 20 else 1

    def epsilon_greedy_action(s):
        player, dealer, ace = s
        if np.random.rand() < epsilon:
            return env.action_space.sample()  # explore
        return int(policy[player, dealer, ace])  # exploit

    for _ in range(num_episodes):
        # --- generate episode following ε-greedy π (no exploring start) ---
        episode = []
        s, info = env.reset()
        done = False

        while not done:
            a = epsilon_greedy_action(s)
            next_s, r, terminated, truncated, info = env.step(a)
            episode.append((s, a, r))
            s = next_s
            done = terminated or truncated

        # --- first-visit MC update of Q ---
        G = 0.0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = gamma * G + r

            earlier = [(episode[i][0], episode[i][1]) for i in range(t)]
            if (s, a) not in earlier:
                player, dealer, ace = s
                counts[player, dealer, ace, a] += 1
                Q[player, dealer, ace, a] += (
                    G - Q[player, dealer, ace, a]
                ) / counts[player, dealer, ace, a]

                # improve: make policy greedy w.r.t. Q
                # (ε-greedy is applied when acting via epsilon_greedy_action)
                policy[player, dealer, ace] = int(np.argmax(Q[player, dealer, ace]))

    return Q, policy


In [ ]:
env = gym.make("Blackjack-v1", sab=True)
Q_on, policy_on = Monte_Carlo_on_policy_control(env, num_episodes=200_000, epsilon=0.1)
plot_policy(policy_on, title="On-policy MC — learned greedy policy")
plot_q_value(Q_on, title=r"On-policy MC — $V(s)=\max_a Q(s,a)$")


## Off-policy Monte Carlo control

Separate **behavior** $\pi'$ ($\varepsilon$-greedy, generates data) from **target** $\pi$ (greedy, optimized).

Uses weighted importance sampling with cumulative weight $W$ and $C(s,a)$.  
After updating $Q$, set $\pi$ greedy; if the behavior action differs from $\pi(s)$, stop the backward pass ($W$ would become $0$).


In [ ]:
def off_policy_MC_control(env, num_episodes, gamma=1.0, epsilon=0.1):
    n_actions = env.action_space.n
    Q = np.zeros((32, 11, 2, n_actions))
    C = np.zeros((32, 11, 2, n_actions))

    # target π: greedy (deterministic), stored as an action
    pi = np.zeros((32, 11, 2), dtype=int)
    for player in range(32):
        for dealer in range(11):
            for ace in range(2):
                pi[player, dealer, ace] = 0 if player >= 20 else 1

    def behavior_action(s):
        """π' = ε-greedy w.r.t. π (covers π)."""
        player, dealer, ace = s
        if np.random.rand() < epsilon:
            return env.action_space.sample()
        return int(pi[player, dealer, ace])

    def target_prob(s, a):
        """π deterministic: 1 if a = π(s), else 0."""
        player, dealer, ace = s
        return 1.0 if a == int(pi[player, dealer, ace]) else 0.0

    def behavior_prob(s, a):
        """π' ε-greedy."""
        player, dealer, ace = s
        a_star = int(pi[player, dealer, ace])
        if a == a_star:
            return 1.0 - epsilon + epsilon / n_actions
        return epsilon / n_actions

    for _ in range(num_episodes):
        # generate an episode with π' (not π)
        episode = []
        s, info = env.reset()
        done = False
        while not done:
            a = behavior_action(s)
            next_s, r, terminated, truncated, info = env.step(a)
            episode.append((s, a, r))
            s = next_s
            done = terminated or truncated

        G = 0.0
        W = 1.0
        for t in range(len(episode) - 1, -1, -1):
            if W == 0.0:
                break
            s, a, r = episode[t]
            player, dealer, ace = s

            G = gamma * G + r
            C[player, dealer, ace, a] += W
            Q[player, dealer, ace, a] += (
                W * (G - Q[player, dealer, ace, a]) / C[player, dealer, ace, a]
            )

            # improve: π greedy
            pi[player, dealer, ace] = int(np.argmax(Q[player, dealer, ace]))

            # if action is no longer π's, future weights are 0
            if a != pi[player, dealer, ace]:
                break

            W *= target_prob(s, a) / behavior_prob(s, a)

    return Q, pi


In [ ]:
env = gym.make("Blackjack-v1", sab=True)
Q_off, policy_off = off_policy_MC_control(env, num_episodes=200_000, epsilon=0.1)
plot_policy(policy_off, title="Off-policy MC — learned target policy")
plot_q_value(Q_off, title=r"Off-policy MC — $V(s)=\max_a Q(s,a)$")


## References

1. S. Airiau. *Apprentissage par renforcement*. M2 IASD, Université Paris Dauphine. Lecture 4: *Résolution à l’aide de méthodes Monte Carlo pour des PDMs épisodiques*. [https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/](https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/)
